# 從 GitHub 更新並執行 CV1

每次開始使用 Colab 時，先執行下一個 cell。第一次會下載 repository，之後會抓取 GitHub 最新版本。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/seanhsu0122/cv1.git'
REPO_DIR = Path('/content/cv1')

if (REPO_DIR / '.git').is_dir():
    print('正在抓取 GitHub 最新版本...')
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
        check=True
    )
else:
    print('第一次使用，正在下載 GitHub repository...')
    subprocess.run(
        ['git', 'clone', REPO_URL, str(REPO_DIR)],
        check=True
    )

os.chdir(REPO_DIR)
print(f'目前工作目錄：{Path.cwd()}')
print('專案檔案：')
for path in sorted(REPO_DIR.iterdir()):
    print(f'  {path.name}')

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

W = 1200
H = 900

image_path = Path('IMG_20261015_191717.jpg')
art_path = Path('ball.png')

if not image_path.exists() or not art_path.exists():
    raise FileNotFoundError(
        '找不到 IMG_20261015_191717.jpg 或 ball.png，請確認兩個檔案已經推送到 GitHub。'
    )

# 將照片中的四個角點校正成正視圖
img = cv2.imread(str(image_path))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

photo_points = np.float32([
    [324, 192],
    [1192, 192],
    [100, 783],
    [1380, 787]
])

normal_points = np.float32([
    [0, 0],
    [1199, 0],
    [0, 899],
    [1199, 899]
])

H_rectify_photo = cv2.getPerspectiveTransform(photo_points, normal_points)
rectified_photo = cv2.warpPerspective(img_rgb, H_rectify_photo, (W, H))

# 將 artwork 變形成照片中畫布的透視形狀，再校正回正視圖
art = cv2.imread(str(art_path))
art = cv2.cvtColor(art, cv2.COLOR_BGR2RGB)
art = cv2.resize(art, (W, H))

photo_points_art = np.float32([
    [324, 192],
    [1192, 193],
    [100, 783],
    [1380, 789]
])

xmin = np.min(photo_points_art[:, 0])
xmax = np.max(photo_points_art[:, 0])
ymin = np.min(photo_points_art[:, 1])
ymax = np.max(photo_points_art[:, 1])

photo_local = np.float32([
    [(324 - xmin) / (xmax - xmin) * (W - 1), (192 - ymin) / (ymax - ymin) * (H - 1)],
    [(1192 - xmin) / (xmax - xmin) * (W - 1), (193 - ymin) / (ymax - ymin) * (H - 1)],
    [(100 - xmin) / (xmax - xmin) * (W - 1), (783 - ymin) / (ymax - ymin) * (H - 1)],
    [(1380 - xmin) / (xmax - xmin) * (W - 1), (789 - ymin) / (ymax - ymin) * (H - 1)]
])

H_pre = cv2.getPerspectiveTransform(normal_points, photo_local)
distorted = cv2.warpPerspective(art, H_pre, (W, H))

H_rectify = cv2.getPerspectiveTransform(photo_local, normal_points)
simulated = cv2.warpPerspective(distorted, H_rectify, (W, H))

plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.imshow(rectified_photo)
plt.title('Rectified original photo')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(art)
plt.title('Desired appearance')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(distorted)
plt.title('Distorted artwork')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(simulated)
plt.title('Simulated view')
plt.axis('off')

plt.show()

distorted_bgr = cv2.cvtColor(distorted, cv2.COLOR_RGB2BGR)
cv2.imwrite('distorted_artwork.png', distorted_bgr)
print('已輸出：distorted_artwork.png')